# Implementasi Gradient Descent dengan TensorFlow dan Keras

Mari kita lihat lebih detail kinerja dari masing-masing metode *Gradient Descent* melalui penulisan kode program. Pada eksperimen ini, kita akan menggunakan *library* **TensorFlow** dan **Keras**.

* **TensorFlow** adalah platform *Machine Learning open-source* yang dikembangkan oleh Google. Platform ini menyediakan *library* dan sumber daya untuk memajukan penelitian di bidang *Machine Learning* dan memudahkan pengembang perangkat lunak dalam membangun aplikasi cerdas. Platform ini utamanya berfokus pada pengembangan dan penggunaan teknologi *Deep Learning*.
* **Keras** merupakan *Application Programming Interface* (API) untuk *Deep Learning* yang ringkas, mudah dipelajari, dan berjalan di atas platform TensorFlow. Dengan kinerja dan skalabilitas tinggi.

Langkah pertama adalah mengimpor *library* yang dibutuhkan dan memastikan ketersediaan *Graphical Processing Unit* (GPU) untuk mempercepat waktu komputasi pelatihan model.

In [4]:
import tensorflow as tf
from tensorflow.keras import datasets
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense


2026-02-24 20:36:53.754520: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-24 20:36:53.761644: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-24 20:36:53.865647: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-24 20:36:54.685071: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

## Memuat dan Menormalisasi Dataset

Selanjutnya, kita akan mengunduh dataset `fashion_mnist`. Dataset ini sudah tersedia secara bawaan di dalam *library* TensorFlow sehingga kita bisa memuatnya dengan sangat mudah. 

Fungsi `load_data()` akan membagi dataset menjadi empat bagian secara otomatis:
1. `X_train`: Data gambar untuk pelatihan (*Training Set*).
2. `y_train`: Label kategori untuk pelatihan.
3. `X_test`: Data gambar untuk pengujian (*Test Set*).
4. `y_test`: Label kategori untuk pengujian.

Untuk memastikan jaringan saraf (NN) dapat belajar dengan performa yang baik dan konvergen lebih cepat, kita perlu melakukan **normalisasi**. Nilai piksel gambar yang aslinya berada pada rentang bilangan bulat 0 hingga 255 akan dibagi dengan $255.0$ agar skalanya berubah menjadi bilangan riil antara 0 dan 1.

In [5]:
# Mengunduh dan memuat dataset fashion_mnist
fashion = datasets.fashion_mnist
(X_train, y_train), (X_test, y_test) = fashion.load_data()

# Normalisasi nilai piksel gambar menjadi di antara 0 dan 1
X_train = X_train / 255.0
X_test  = X_test / 255.0

print(f"Dimensi X_train: {X_train.shape}")
print(f"Dimensi X_test: {X_test.shape}")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 19s 1us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 4s 1us/step
Dimensi X_train: (60000, 28, 28)
Dimensi X_test: (10000, 28, 28)


## Membangun Arsitektur Jaringan Saraf (Neural Network)

Kita akan menyiapkan model NN sesuai dengan arsitektur yang telah ditetapkan. Terdapat dua cara membangun model di Keras: *Sequential Model* (berurutan) dan *Functional API*. Pada eksperimen ini, kita menggunakan *Sequential Model* dengan menambahkan layer secara bertahap menggunakan metode `add()`.

**Konfigurasi Arsitektur:**
1. **Input Layer (`Flatten`)**: Mengubah matriks gambar 2D berukuran $28 \times 28$ piksel menjadi vektor 1D berukuran $784$ elemen.
2. **Hidden Layer (`Dense`)**: Lapisan tersembunyi dengan 32 neuron (*node*) yang terhubung penuh (*fully connected*). Lapisan ini menggunakan fungsi aktivasi **ReLU** (*Rectified Linear Unit*).
3. **Output Layer (`Dense`)**: Lapisan keluaran dengan 10 neuron, merepresentasikan 10 kategori pakaian pada dataset. Lapisan ini menggunakan fungsi aktivasi **Softmax**. Fungsi Softmax akan memastikan keluaran model berupa nilai probabilitas antara 0 dan 1, di mana total jumlah probabilitas dari ke-10 neuron tersebut adalah tepat 1.

Setelah struktur dibuat, model harus dikompilasi (`compile`).
* **Loss Function**: Menggunakan `sparse_categorical_crossentropy` karena ini adalah masalah klasifikasi multikelas (*Multiclass Classification*) dan label berupa angka (*integer* 0-9), bukan representasi *one-hot encoding*.
* **Optimizer**: Menggunakan `SGD` (*Stochastic Gradient Descent*).
* **Metrics**: Menggunakan `accuracy` untuk memantau tingkat akurasi selama pelatihan.

In [ ]:
from keras.models import Sequential
from keras.layers import Input, Flatten, Dense

# Membangun arsitektur Sequential Model
model = Sequential([
    Input(shape=(28, 28)),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
])

# Kompilasi model
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='sgd',
    metrics=['accuracy']
)

## Inspeksi Parameter Jaringan

Mari kita lihat struktur jaringan yang telah dikompilasi menggunakan fungsi `summary()`.

Perhitungan matematis jumlah parameter:
* **Input Layer**: Berukuran $784$ (dari $28 \times 28$). Tidak memiliki parameter bobot untuk dilatih, sehingga nilainya $0$.
* **Hidden Layer**: Memiliki 32 neuron. Menerima $784$ input. 
  Total bobot dan bias = $(784 \times 32) + 32 = 25.120$ parameter.
* **Output Layer**: Memiliki 10 neuron. Menerima 32 input dari *hidden layer*.
  Total bobot dan bias = $(32 \times 10) + 10 = 330$ parameter.

Secara keseluruhan, algoritme *Gradient Descent* harus mencari dan mengoptimalkan **25.450 parameter** yang bisa dilatih (*trainable params*). Kata `None` pada *Output Shape* mengindikasikan ukuran *batch* yang bersifat fleksibel dan akan ditentukan saat proses pelatihan.

In [9]:
# Menampilkan ringkasan arsitektur dan total parameter
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │        25,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,450 (99.41 KB)

 Trainable params: 25,450 (99.41 KB)

 Non-trainable params: 0 (0.00 B)

## Pelatihan Model (Training)

Langkah selanjutnya adalah melatih model. Pada modul ini, ditekankan perbedaan performa berdasarkan pengaturan *batch size*. Kita akan menyimulasikan **SGD_32** (Mini-Batch SGD dengan *batch size* = 32).

Parameter pelatihan:
* `epochs=50`: Model akan melihat seluruh dataset sebanyak 50 kali putaran.
* `batch_size=32`: Bobot jaringan akan diperbarui setiap kali model memproses 32 contoh gambar.
* `validation_split=0.2`: Secara otomatis menyisihkan 20% dari data *Training Set* untuk dijadikan *Validation Set*. Data validasi ini berfungsi untuk mengevaluasi performa model (*val_loss* dan *val_accuracy*) di setiap akhir putaran (*epoch*) untuk memantau terjadinya *overfitting* atau *underfitting*.

In [10]:
# Melatih model dengan batch_size 32 (Mini-batch SGD) selama 50 epochs
# Catatan: Untuk demonstrasi cepat di sel ini, Anda bisa mengubah epochs menjadi 5 atau 10.
history = model.fit(
    X_train, 
    y_train, 
    epochs=50, 
    validation_split=0.2, 
    batch_size=32
)

Epoch 1/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 790us/step - accuracy: 0.7085 - loss: 0.8758 - val_accuracy: 0.7938 - val_loss: 0.6098
Epoch 2/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 695us/step - accuracy: 0.8094 - loss: 0.5635 - val_accuracy: 0.8163 - val_loss: 0.5230
Epoch 3/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 687us/step - accuracy: 0.8251 - loss: 0.5052 - val_accuracy: 0.8225 - val_loss: 0.4937
Epoch 4/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 698us/step - accuracy: 0.8361 - loss: 0.4750 - val_accuracy: 0.8307 - val_loss: 0.4705
Epoch 5/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 686us/step - accuracy: 0.8419 - loss: 0.4549 - val_accuracy: 0.8410 - val_loss: 0.4514
Epoch 6/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 691us/step - accuracy: 0.8472 - loss: 0.4402 - val_accuracy: 0.8421 - val_loss: 0.4435
Epoch 7/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 686us/step - accuracy: 0.8511 - loss: 0.4267 - val_accuracy: 0.8487 - val_loss: 0.4293
Epoch 8/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1s 688us/step - accuracy: 0.8550 -

## Evaluasi Performa Model

Setelah pelatihan selesai, kita harus menguji kemampuan model terhadap data yang benar-benar baru dan belum pernah dilihat oleh model sebelumnya, yaitu `X_test`. Fungsi `evaluate()` akan mengembalikan dua nilai: **Loss** akhir dan **Akurasi** akhir.

Idealnya, akurasi pada *Test Set* tidak akan berbeda jauh dengan *val_accuracy* pada tahap pelatihan, yang menunjukkan bahwa model memiliki kemampuan generalisasi yang baik. Berdasarkan literatur, akurasi model ini akan berada di kisaran ~83%.

In [11]:
# Mengevaluasi model menggunakan Test Set
test_loss, test_acc = model.evaluate(X_test, y_test)

print(f"\nLoss pada Test Set: {test_loss:.4f}")
print(f"Akurasi pada Test Set: {test_acc:.4f} ({(test_acc*100):.2f}%)")

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 589us/step - accuracy: 0.8696 - loss: 0.3774

Loss pada Test Set: 0.3774
Akurasi pada Test Set: 0.8696 (86.96%)
